In [ ]:
import os
import cv2
import numpy as np
from glob import glob

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
WIRE_DIR      = "/content/drive/MyDrive/cnn_dataset/walls"      # your wireframe .png
MASK_DIR      = "/content/drive/MyDrive/cnn_dataset/colors"      # your segmentation masks (integer IDs)
FURN_DIR      = "/content/drive/MyDrive/cnn_dataset/plans"   # your furnished floorplans
OUT_WIRE   = "/content/drive/MyDrive/cnn_dataset/walls_segmented"       # where to save wireframe crops
OUT_FURN   = "/content/drive/MyDrive/cnn_dataset/plans_segmented"      # where to save furnished crops

os.makedirs(OUT_WIRE, exist_ok=True)
os.makedirs(OUT_FURN, exist_ok=True)

In [ ]:
color_map = {
    (  0,   0,   0): 0,
    (255, 154,   0): 1,
    ( 50,  99, 155): 2,
    (  0, 155, 255): 3,
    (155, 255,   0): 4,
    (100,  80,  71): 5
}

In [ ]:
class_names = ['background','bedroom','hallway','bathroom','living','empty-room']
# create a subdirectory per class (skip background)
for cls_idx, cls_name in enumerate(class_names):
    if cls_idx == 0:  # skip background
        continue
    os.makedirs(os.path.join(OUT_WIRE, cls_name), exist_ok=True)
    os.makedirs(os.path.join(OUT_FURN, cls_name), exist_ok=True)

In [ ]:
# List all examples by basename
wire_files = sorted(glob(os.path.join(WIRE_DIR,  "*.png")))
mask_files = sorted(glob(os.path.join(MASK_DIR,  "*.png")))
furn_files = sorted(glob(os.path.join(FURN_DIR,  "*.png")))

In [ ]:
assert len(wire_files)==len(mask_files)==len(furn_files)
base_names = [os.path.basename(p)[:-4] for p in wire_files]

In [ ]:
print(len(base_names))

357


In [ ]:
# for name in base_names:
#     # load
#     wire = cv2.imread(os.path.join(WIRE_DIR,  name + ".png"), cv2.IMREAD_UNCHANGED)
#     mask = cv2.imread(os.path.join(MASK_DIR,  name + ".png"), cv2.IMREAD_UNCHANGED)
#     furn = cv2.imread(os.path.join(FURN_DIR,  name + ".png"), cv2.IMREAD_UNCHANGED)

#     # if masks are RGB, convert to class IDs first:
#     # mask = rgb_to_class(cv2.cvtColor(mask, cv2.COLOR_BGR2RGB), PALETTE)

#     # ensure mask is single‐channel int IDs
#     if mask.ndim==3:
#         mask = mask[:,:,0]

#     # for each room class, find connected components
#     for cls in np.unique(mask):
#         if cls==0:  # skip background if that’s index 0
#             continue

#         # binary mask of this class
#         binm = (mask == cls).astype(np.uint8)

#         # find connected components of this class
#         n_lbl, labels, stats, centroids = cv2.connectedComponentsWithStats(binm, connectivity=8)

#         for lbl in range(1, n_lbl):
#             area = stats[lbl, cv2.CC_STAT_AREA]
#             if area < 100:          # skip tiny specks; adjust threshold
#                 continue

#             # get bbox
#             x, y, w, h, _ = stats[lbl]
#             region_mask = (labels == lbl)

#             # crop both wire and furn
#             wire_crop = wire[y:y+h, x:x+w].copy()
#             furn_crop = furn[y:y+h, x:x+w].copy()

#             # apply the region_mask as alpha:
#             #   wireroom: keep white lines, zero elsewhere
#             alpha = region_mask[y:y+h, x:x+w].astype(np.uint8)
#             for c in range(wire_crop.shape[2]):
#                 wire_crop[:,:,c] = wire_crop[:,:,c] * alpha

#             #   furnroom: keep only pixels inside the room; black outside
#             for c in range(furn_crop.shape[2]):
#                 furn_crop[:,:,c] = furn_crop[:,:,c] * alpha

#             # save
#             out_inp = os.path.join(OUT_INP_DIR,  f"{name}_cls{cls}_part{lbl}.png")
#             out_tgt = os.path.join(OUT_TGT_DIR,  f"{name}_cls{cls}_part{lbl}.png")
#             cv2.imwrite(out_inp, wire_crop)
#             cv2.imwrite(out_tgt, furn_crop)

#     print(f"Extracted rooms for {name}")

In [ ]:
for name in base_names:
    # if name not in mask_files or name not in furn_files:
    #     continue
    wire = cv2.imread(os.path.join(WIRE_DIR,  name + ".png"), cv2.IMREAD_UNCHANGED)
    mask = cv2.imread(os.path.join(MASK_DIR,  name + ".png"), cv2.IMREAD_UNCHANGED)
    furn = cv2.imread(os.path.join(FURN_DIR,  name + ".png"), cv2.IMREAD_UNCHANGED)
    # wire = cv2.imread(wf)
    # mask = cv2.imread(mask_files[name])
    # furn = cv2.imread(furn_files[name])

    # convert RGB mask → single-channel IDs
    rgb = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
    mask_id = np.zeros(rgb.shape[:2], dtype=np.uint8)
    for color, idx in color_map.items():
        match = np.all(rgb == color, axis=-1)
        mask_id[match] = idx

    H, W = mask_id.shape

    # for each room class (skip background=0)
    for cls in np.unique(mask_id):
        if cls == 0:
            continue

        # build binary mask of this class
        binm = (mask_id == cls).astype(np.uint8)
        n_lbl, labels, stats, _ = cv2.connectedComponentsWithStats(binm, connectivity=8)

        for lbl in range(1, n_lbl):
            area = stats[lbl, cv2.CC_STAT_AREA]
            if area < 100:    # skip tiny specks
                continue

            # make a boolean keep-mask for this region
            keep = (labels == lbl)    # shape (H, W) bool

            if furn.shape[2] == 4:
              furn = furn[:, :, :3]   # drop alpha, keep BGR

            # expand to 3-channels
            keep3 = np.repeat(keep[:, :, None], 3, axis=2)

            # zero-out everything outside the room
            wire_masked = np.where(keep3, wire, 0)
            furn_masked = np.where(keep3, furn, 0)

            # save full-size masked images
            stem = name[4:]
            print(stem)
            # out_w = os.path.join(OUT_WIRE, f"{stem}_cls{cls}_part{lbl}.png")
            # out_f = os.path.join(OUT_FURN, f"{stem}_cls{cls}_part{lbl}.png")
            cls_name = class_names[cls]  # get the folder name
            out_w = os.path.join(OUT_WIRE, cls_name, f"{stem}_{cls_name}_{lbl}.png")
            out_f = os.path.join(OUT_FURN, cls_name, f"{stem}_{cls_name}_{lbl}.png")

            cv2.imwrite(out_w, wire_masked)
            cv2.imwrite(out_f, furn_masked)

    print(f"Masked rooms for {name}")